# TrackFormer v17 — a loss that actually constrains the track

Two findings drive this version.

**1. The v15/v16 notebook lost half the track loss.** v13/v14/v14.1 score position twice — per-step
displacement (weighted by `sqrt(lead)`) *and* the cumulative position. The notebook kept only the
per-step term:

    # v14.1                                    # v15/v16 notebook
    step = smooth_l1(pm, tm) * LEADW           d = (pm - tm) * mask
    pos  = smooth_l1(cumsum(pm), cumsum(tm))   return |d|
    return step + pos                          # <- no cumulative term at all

Nothing penalised *where the storm ended up*, only how each 6-hour hop looked in isolation. Errors
that individually look small are free to accumulate in the same direction.

**2. Speed and heading were never scored as such.** A step vector that is the right length in the
wrong direction costs the same as one the wrong length in the right direction. For a track forecast
those are not equivalent errors — direction compounds, magnitude does not.

v17 scores all four:

| term | what it constrains |
|---|---|
| `step` | per-6h displacement, weighted by sqrt(lead) — restored from v14.1 |
| `pos` | **cumulative position** — restored; this is what stops long-lead drift |
| `spd` | magnitude of each step: forward speed |
| `dir` | 1 - cos(angle) between predicted and observed step: heading |

**SST is dropped.** The v16 ablation — same architecture, same seeds, SST channel zeroed — found
removing it *improved* track from 479.13 to **466.30 km** while helping pressure by only 0.19 hPa.
v17 therefore runs on 4 channels, and 466.30 km is the baseline it must beat.

**5 seeds instead of 3**, so the RMT ensemble combination that follows has enough estimators to
work with.

## Upload these to Google Drive first (`MyDrive/typhoon/`)
- `track_windows_v13.npz` (43 MB)
- `steer5_int8.npz` (90 MB) — ch4 is simply not read

**Runtime -> Change runtime type -> L4 GPU.**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/typhoon'
!ls -la {DATA}
# copy to local disk -- Drive I/O is slow and we touch these every epoch
!mkdir -p /content/d && cp {DATA}/track_windows_v13.npz {DATA}/steer5_int8.npz /content/d/

## Data + equatorial-mirror augmentation

In [ ]:
import numpy as np, torch, math, json, time, os, shutil
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda")
TARGET_SCALE = torch.tensor([100., 100., 35., 20., 50.] + [50.] * 12, device=DEVICE)
EPOCHS, PATIENCE, BATCH = 80, 14, 512
LR, WEIGHT_DECAY = 3e-4, 5e-2
STEER_DROP, STEER_CLIP = 0.20, 4.0
MIRROR_P = 0.5          # probability of mirroring a training sample

KIN_COLS = [0, 1, 2, 3, 21, 22, 23, 40, 41, 42, 43]
THERMO_COLS = [4, 5, 6, 7] + list(range(8, 20)) + list(range(24, 40)) + [44, 45, 46, 47]
ENV_COLS = [48, 49, 50, 51, 52, 53]
KIN_DIM, THERMO_DIM, ENV_DIM = len(KIN_COLS), len(THERMO_COLS), len(ENV_COLS)

z = np.load("/content/d/track_windows_v13.npz", allow_pickle=True)
track = z["track"].astype("float32"); target = z["target"].astype("float32")
mask = z["target_mask"].astype("float32"); years = z["year"].astype(int)
sids = z["storm_id"].astype(str); basins = z["basin"].astype(str)
tmean = z["track_mean"].astype("float32"); tstd = z["track_std"].astype("float32")
# 5 channels: SLPanom, SLPtend, u500, v500, SST-28C -- all on the same storm-centred 2.5 deg
# grid. int8 in sigma units (max quantization error 0.016 sigma, and we clip at 4 sigma anyway):
# 560 MB of float16 becomes 90 MB with no meaningful loss.
_q = np.load("/content/d/steer5_int8.npz")
# ch4 is SST. The v16 ablation showed it costs 12.8 km of track error and buys 0.19 hPa of
# pressure, so v17 reads only the four fields that earn their place.
SLP = np.clip(_q["q"][:, :4].astype("float32") / 31.75, -STEER_CLIP, STEER_CLIP)
AVAIL = _q["ok"]        # [N,3] per group: SLP pair, steering pair, SST (ch4 unused here)
del _q
print(f"field availability  SLP {AVAIL[:,0].mean():.3f}  steer {AVAIL[:,1].mean():.3f}  "
      f"SST {AVAIL[:,2].mean():.3f}   (unavailable == exact zeros, not fabricated)")
v0_raw = track[:, -1, 2:4] * tstd[2:4] + tmean[2:4]
vp_raw = track[:, -2, 2:4] * tstd[2:4] + tmean[2:4]
vpair = np.concatenate([v0_raw, vp_raw], axis=1).astype("float32")
print("windows", len(track))

# ---- mirror index maps -------------------------------------------------
# raw track columns that simply negate under lat -> -lat
NEG_T = [1, 3, 21, 22, 41, 43, 48]        # n, sn, sin/cos season, heading-north, turn, lat
# wind-radius quadrant order is (ne, se, sw, nw) per threshold; N<->S swaps ne<->se and nw<->sw
def quad_perm(base):
    p = list(range(base, base + 12))
    for t in range(3):
        o = base + 4 * t
        p[o - base + 0], p[o - base + 1] = o + 1, o + 0     # ne <-> se
        p[o - base + 2], p[o - base + 3] = o + 3, o + 2     # sw <-> nw
    return p
RAD_T = quad_perm(8)                       # track radii cols 8..19
FLG_T = quad_perm(28)                      # their validity flags 28..39
RAD_Y = [5 + i for i in range(12)]         # target radii 5..16
RAD_Y_PERM = [5 + (q - 8) for q in quad_perm(8)]

tm_t = torch.from_numpy(tmean); ts_t = torch.from_numpy(tstd)

def mirror(tr, tg, mk, sp):
    """tr [H,54] normalized, tg [L,17] raw, mk [L,17], sp [4,17,17]. Returns mirrored copies."""
    raw = tr * ts_t + tm_t                                   # de-normalize to flip signs
    raw = raw.clone()
    raw[:, NEG_T] = -raw[:, NEG_T]
    raw[:, 8:20] = raw[:, RAD_T]
    raw[:, 28:40] = raw[:, FLG_T]
    tr2 = (raw - tm_t) / ts_t
    tg2 = tg.clone(); tg2[:, 1] = -tg2[:, 1]                 # north displacement
    tg2[:, 5:17] = tg[:, RAD_Y_PERM]
    mk2 = mk.clone(); mk2[:, 5:17] = mk[:, RAD_Y_PERM]
    sp2 = torch.flip(sp, dims=[1]).clone()                   # mirror the field north-south
    sp2[3] = -sp2[3]                                         # v500 is a signed northward wind
    #  ch0/1 SLP and ch4 SST are scalar fields: they flip but keep their sign
    return tr2, tg2, mk2, sp2


class DS(Dataset):
    def __init__(self, idx, aug): self.idx = np.asarray(idx); self.aug = aug
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = int(self.idx[i])
        tr = torch.from_numpy(track[j]); tg = torch.from_numpy(target[j])
        mk = torch.from_numpy(mask[j]);  sp = torch.from_numpy(SLP[j])
        vp = torch.from_numpy(vpair[j])
        if self.aug and torch.rand(()) < MIRROR_P:
            tr, tg, mk, sp = mirror(tr, tg, mk, sp)
            vp = vp.clone(); vp[1] = -vp[1]; vp[3] = -vp[3]   # north components of v0 / vprev
        return tr, vp, sp, tg, mk

fy = {s: int(years[sids == s].min()) for s in np.unique(sids)}
tr_idx = np.where(np.isin(sids, [s for s, y in fy.items() if y <= 2015]))[0]
va_idx = np.where(np.isin(sids, [s for s, y in fy.items() if 2016 <= y <= 2019]))[0]
te_idx = np.where(np.isin(sids, [s for s, y in fy.items() if y >= 2020]))[0]
print(f"split train={len(tr_idx)} valid={len(va_idx)} test={len(te_idx)}  (train effectively 2x with mirror)")

def loader(idx, sh, aug=False):
    return DataLoader(DS(idx, aug), batch_size=BATCH, shuffle=sh, num_workers=2,
                      pin_memory=True, persistent_workers=True, drop_last=sh)

### Sanity-check the mirror (a mirrored storm must be a *valid* storm)

In [ ]:
# A mirrored sample must: negate lat, preserve speed/vmax, and preserve |displacement|.
j = int(tr_idx[0])
tr = torch.from_numpy(track[j]); tg = torch.from_numpy(target[j])
mk = torch.from_numpy(mask[j]); sp = torch.from_numpy(SLP[j])
tr2, tg2, mk2, sp2 = mirror(tr, tg, mk, sp)
raw  = (tr  * ts_t + tm_t).numpy(); raw2 = (tr2 * ts_t + tm_t).numpy()
print(f"lat        {raw[-1,48]:+7.2f} -> {raw2[-1,48]:+7.2f}   (must negate)")
print(f"speed      {raw[-1,42]:7.2f} -> {raw2[-1,42]:7.2f}   (must match)")
print(f"vmax       {raw[-1,4]:7.2f} -> {raw2[-1,4]:7.2f}   (must match)")
print(f"|disp|     {np.hypot(*raw[-1,:2]):7.2f} -> {np.hypot(*raw2[-1,:2]):7.2f}   (must match)")
print(f"u500 mean  {sp[2].mean():+7.3f} -> {sp2[2].mean():+7.3f}   (must match)")
print(f"v500 mean  {sp[3].mean():+7.3f} -> {sp2[3].mean():+7.3f}   (must negate)")
assert torch.allclose(sp[2].mean(), sp2[2].mean(), atol=1e-4), "u500 is unchanged under N-S mirror"
assert abs(raw[-1,48] + raw2[-1,48]) < 1e-2 and abs(raw[-1,42] - raw2[-1,42]) < 1e-2
assert torch.allclose(mirror(*mirror(tr, tg, mk, sp))[0], tr, atol=1e-4), "mirror must be an involution"
print("\nOK - mirror is physically consistent and self-inverse")

## Model (v14.1 architecture: steering dropout + clipping)

In [ ]:
def sinusoidal(n, d):
    p = torch.arange(n).unsqueeze(1).float()
    dv = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
    e = torch.zeros(n, d); e[:, 0::2] = torch.sin(p * dv); e[:, 1::2] = torch.cos(p * dv)
    return e

def enc(d, h, ffn, dr, depth):
    return nn.TransformerEncoder(nn.TransformerEncoderLayer(d, h, ffn, dr, batch_first=True,
                                 norm_first=True, activation="gelu"), depth)

def dec(d, h, ffn, dr, depth):
    return nn.TransformerDecoder(nn.TransformerDecoderLayer(d, h, ffn, dr, batch_first=True,
                                 norm_first=True, activation="gelu"), depth)


class TrackFormerV17(nn.Module):
    def __init__(self, d=256, h=8, ffn=1024, dr=0.15, hist=9, leads=20):
        super().__init__()
        self.leads = leads
        self.kin_proj = nn.Linear(KIN_DIM, d); self.thermo_proj = nn.Linear(THERMO_DIM, d)
        self.env_proj = nn.Linear(ENV_DIM, d)
        self.register_buffer("kin_time", sinusoidal(hist, d).unsqueeze(0))
        self.register_buffer("thermo_time", sinusoidal(hist, d).unsqueeze(0))
        self.register_buffer("env_time", sinusoidal(hist, d).unsqueeze(0))
        self.kin_enc = enc(d, h, ffn, dr, 3); self.thermo_enc = enc(d, h, ffn, dr, 3)
        self.env_enc = enc(d, h, ffn, dr, 2)
        self.track_dec = dec(d, h, ffn, dr, 3); self.int_dec = dec(d, h, ffn, dr, 3)
        self.track_q = nn.Parameter(torch.randn(1, leads, d) * 0.02)
        self.int_q = nn.Parameter(torch.randn(1, leads, d) * 0.02)
        self.register_buffer("qpos", sinusoidal(leads, d))
        self.adapter = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, d))
        nn.init.zeros_(self.adapter[-1].weight); nn.init.zeros_(self.adapter[-1].bias)
        self.alpha = nn.Parameter(torch.zeros(leads)); self.rho = nn.Parameter(torch.ones(leads))
        self.gturn = nn.Parameter(torch.zeros(leads))
        self.steer_cnn = nn.Sequential(
            nn.Conv2d(4, 24, 3, padding=1), nn.GELU(), nn.Dropout2d(0.10),
            nn.Conv2d(24, 48, 3, stride=2, padding=1), nn.GELU(), nn.Dropout2d(0.10),
            nn.Conv2d(48, d, 3, stride=2, padding=1), nn.GELU())
        self.steer_pos = nn.Parameter(torch.zeros(1, 25, d))
        self.track_res = nn.Linear(d, 2)
        nn.init.zeros_(self.track_res.weight); nn.init.zeros_(self.track_res.bias)
        self.int_state = nn.Linear(d, 15); self.int_logscale = nn.Linear(d, 15)

    def forward(self, track, vpair, slp):
        b = track.shape[0]
        kin = self.kin_enc(self.kin_proj(track[:, :, KIN_COLS]) + self.kin_time)
        thermo = self.thermo_enc(self.thermo_proj(track[:, :, THERMO_COLS]) + self.thermo_time)
        env = self.env_enc(self.env_proj(track[:, :, ENV_COLS]) + self.env_time)
        if self.training and STEER_DROP > 0:
            keep = (torch.rand(b, 1, 1, 1, device=slp.device) >= STEER_DROP).float()
            slp = slp * keep
        st = self.steer_cnn(slp).flatten(2).transpose(1, 2) + self.steer_pos
        tq = (self.track_q + self.qpos.unsqueeze(0)).expand(b, -1, -1)
        h_track = self.track_dec(tq, torch.cat([kin, env, st], dim=1))
        h_track = h_track + self.alpha.view(1, self.leads, 1) * self.adapter(thermo.mean(1).detach()).unsqueeze(1)
        # CURVED baseline: extrapolate the current (damped) turn rate instead of a straight line.
        v0, vp = vpair[:, :2], vpair[:, 2:]
        s0 = v0.norm(dim=1, keepdim=True).clamp(min=1e-3)                  # current speed [b,1]
        phi0 = torch.atan2(v0[:, 1], v0[:, 0])                             # current heading [b]
        dphi = phi0 - torch.atan2(vp[:, 1], vp[:, 0])
        omega = torch.atan2(torch.sin(dphi), torch.cos(dphi))              # wrapped turn rate [b]
        phil = phi0.unsqueeze(1) + self.gturn.view(1, self.leads) * omega.unsqueeze(1)   # [b,L]
        speed = self.rho.view(1, self.leads) * s0                          # damped speed [b,L]
        base = torch.stack([speed * torch.cos(phil), speed * torch.sin(phil)], dim=-1) / 100.0
        motion = base + self.track_res(h_track)
        iq = (self.int_q + self.qpos.unsqueeze(0)).expand(b, -1, -1)
        h_int = self.int_dec(iq, torch.cat([thermo, env, kin.detach(), st.detach()], dim=1))
        istate = self.int_state(h_int); ilog = self.int_logscale(h_int)
        return torch.cat([motion, istate], -1), torch.cat([torch.zeros_like(motion), ilog], -1)

## Loss + train loop (T4 mixed precision)

In [ ]:
# long leads are where the money is, and they are the minority of the loss by default
LEADW = torch.sqrt(torch.arange(1, 21, device=DEVICE).float()); LEADW = LEADW / LEADW.mean()
W_SPD = float(os.environ.get("W_SPD", "0.30"))     # forward-speed magnitude
W_DIR = float(os.environ.get("W_DIR", "0.30"))     # heading
EPS = 1e-3

def track_loss(s, tn, m):
    """Four terms: per-step, cumulative position, forward speed, heading.

    v15/v16 had only the first. Nothing constrained where the storm ENDED UP, so small same-signed
    per-step errors were free to accumulate over 20 leads. Speed and heading are scored explicitly
    because a step of the right length in the wrong direction and one of the wrong length in the
    right direction are not equally costly for a track: direction compounds, magnitude does not.
    """
    pm, tm, mm = s[..., :2], tn[..., :2], m[..., :2]
    w = mm * LEADW.view(1, 20, 1)
    step = (F.smooth_l1_loss(pm, tm, reduction="none") * w).sum() / w.sum().clamp(min=1)
    pos = (F.smooth_l1_loss(torch.cumsum(pm, 1), torch.cumsum(tm, 1), reduction="none")
           * mm).sum() / mm.sum().clamp(min=1)
    ps = torch.sqrt((pm ** 2).sum(-1) + 1e-8)          # predicted step length
    ts = torch.sqrt((tm ** 2).sum(-1) + 1e-8)          # observed step length
    mv = mm[..., 0]
    spd = (F.smooth_l1_loss(ps, ts, reduction="none") * mv).sum() / mv.sum().clamp(min=1)
    cos = (pm * tm).sum(-1) / (ps.clamp(min=EPS) * ts.clamp(min=EPS))
    # heading is undefined for a storm that has not moved -- do not train on it
    hv = mv * (ts > EPS).float()
    dirl = ((1.0 - cos) * hv).sum() / hv.sum().clamp(min=1)
    return step + pos + W_SPD * spd + W_DIR * dirl

def int_loss(s, logs, tn, m):
    d = (s[..., 2:] - tn[..., 2:]).abs()
    lg = logs[..., 2:].clamp(-4, 4)
    nll = (d * torch.exp(-lg) + lg) * m[..., 2:]
    return nll.sum() / m[..., 2:].sum().clamp(min=1)

def total_loss(s, logs, tgt, m):
    tn = tgt / TARGET_SCALE
    return track_loss(s, tn, m) + int_loss(s, logs, tn, m)


DRIVE_OUT = "/content/drive/MyDrive/typhoon"

def train_one(seed, ckpt):
    torch.manual_seed(seed); np.random.seed(seed)
    model = TrackFormerV17().to(DEVICE)
    print(f"seed {seed} | params {sum(p.numel() for p in model.parameters()):,}")
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler()
    tl = loader(tr_idx, True, aug=True); vl = loader(va_idx, False)

    def run(ld, train):
        model.train(train); tot = cnt = 0
        for tr, v0, sp, tg, m in ld:
            tr, v0, sp, tg, m = [x.to(DEVICE, non_blocking=True) for x in (tr, v0, sp, tg, m)]
            with torch.set_grad_enabled(train), torch.cuda.amp.autocast():
                s, ls = model(tr, v0, sp); loss = total_loss(s, ls, tg, m)
            if train:
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update()
            tot += float(loss.detach()) * len(tr); cnt += len(tr)
        return tot / cnt

    best, bad, t0 = 1e9, 0, time.time()
    for ep in range(EPOCHS):
        te = time.time(); trl = run(tl, True)
        with torch.no_grad(): vv = run(vl, False)
        sched.step()
        if vv < best:
            best, bad = vv, 0
            torch.save({"model": model.state_dict(), "epoch": ep, "best_val": best,
                        "track_mean": tmean, "track_std": tstd}, ckpt)
            # Colab VMs die on idle/12h timeout -- persist every improvement, not just at the end
            try: shutil.copy(ckpt, DRIVE_OUT)
            except Exception as e: print("  (drive copy failed:", e, ")")
        else:
            bad += 1
        print(f"ep {ep:03d} | train {trl:.5f} | val {vv:.5f} | best {best:.5f} | {time.time()-te:.0f}s", flush=True)
        if bad >= PATIENCE:
            print("early stop", ep); break
    print(f"done in {(time.time()-t0)/60:.1f} min | best_val {best:.5f}")
    return best

## Evaluation — identical harness to the local one (validated to 0.04 km)

In [ ]:
@torch.no_grad()
def metrics(models, idx):
    """Averages the motion of an ENSEMBLE of models (seed ensembling)."""
    P, T, M = [], [], []
    for tr, v0, sp, tg, m in loader(idx, False):
        tr, v0, sp = tr.to(DEVICE), v0.to(DEVICE), sp.to(DEVICE)
        s = torch.stack([mm(tr, v0, sp)[0] for mm in models]).mean(0)
        P.append((s * TARGET_SCALE).float().cpu().numpy()); T.append(tg.numpy()); M.append(m.numpy())
    P, T, M = np.concatenate(P), np.concatenate(T), np.concatenate(M); out = {}
    pt, tt = np.cumsum(P[..., :2], 1), np.cumsum(T[..., :2], 1)     # targets are PER-STEP; cumsum matters
    out["track_error_km"] = round(float(np.sqrt(((pt - tt) ** 2).sum(-1)).mean()), 2)
    for i, nm in [(2, "vmax_mae_kt"), (3, "pressure_mae_hpa"), (4, "rmw_mae_km")]:
        v = M[..., i] > 0.5
        out[nm] = round(float(np.abs(P[..., i][v] - T[..., i][v]).mean()), 2) if v.any() else None
    rm = M[..., 5:17] > 0.5
    out["radius_mae_km"] = round(float(np.abs(P[..., 5:17] - T[..., 5:17])[rm].mean()), 2)
    return out

def load_model(ckpt):
    m = TrackFormerV17().to(DEVICE).eval()
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=False)["model"])
    return m

## Run: 3 seeds, then ensemble

Seed-ensembling is the cheapest reliable gain in trajectory forecasting (typically 5-10%) —
independent seeds make *different* errors, and averaging cancels them. On a T4 each seed is
roughly 1-2 min/epoch, so three seeds is well under an hour.

In [ ]:
CKPTS = []
for seed in (0, 1, 2, 3, 4):
    c = f"/content/v17_seed{seed}.pt"
    train_one(seed, c); CKPTS.append(c)   # already mirrored to Drive after every improvement

In [ ]:
full = z["n_leads"].astype(int) == 20
# WP+EP is the front being tested; AB keeps the all-basin number for reference
wpep = np.array([i for i in te_idx if full[i] and basins[i] in ("WP", "EP")])
wp   = np.array([i for i in te_idx if full[i] and basins[i] == "WP"])
ab   = np.array([i for i in te_idx if full[i]])
models = [load_model(c) for c in CKPTS]

# baselines re-measured on the REPAIRED fields, same harness (WP+EP 2020+, all-lead mean km)
print("BASELINES [WP+EP 2020+, repaired]  v14 489.3 | v14.1 500.2 | v16 479.1 | v16-noSST 466.3  <- the one to beat")
print("BASELINES (WP only)         v10 614.19 | v13 618.66 | v14 545.74 | v15ens 519.09\n")
for i, c in enumerate(CKPTS):
    print(f"v16 seed{i}  WP+EP:", json.dumps(metrics([load_model(c)], wpep)))
print("\nv16 ENSEMBLE WP+EP:", json.dumps(metrics(models, wpep)))
print("v16 ENSEMBLE WP   :", json.dumps(metrics(models, wp)))
print("v16 ENSEMBLE ALL  :", json.dumps(metrics(models, ab)))

### Per-storm check — Bavi against the *repaired* references

In [ ]:
sid_all = z["storm_id"].astype(str); nl = z["n_leads"].astype(int)
print("120h track error, mean over that storm's windows:")
print(f"{'storm':12s} {'v13':>8} {'v14fix':>8} {'v141fix':>8} {'v17ens':>8}")
# v14/v14.1 re-measured on the repaired fields (Bavi was 3442/2257 on the fabricated ones)
REF = {"Bavi": (639, 1539, 797), "Wayne": (628, 730, 739),
       "Co-may": (1098, 941, 905), "Hinnamnor": (1260, 1095, 1022)}
for s, nm in [("2026182N09163","Bavi"),("1986228N19120","Wayne"),
              ("2025203N20124","Co-may"),("2022239N22150","Hinnamnor")]:
    k = np.where((sid_all == s) & (nl == 20))[0]
    if not len(k): continue
    P, T = [], []
    with torch.no_grad():
        for tr, v0, sp, tg, m in loader(k, False):
            tr, v0, sp = tr.to(DEVICE), v0.to(DEVICE), sp.to(DEVICE)
            s_ = torch.stack([mm(tr, v0, sp)[0] for mm in models]).mean(0)
            P.append((s_ * TARGET_SCALE).float().cpu().numpy()); T.append(tg.numpy())
    P, T = np.concatenate(P), np.concatenate(T)
    e = np.sqrt(((np.cumsum(P[..., :2], 1) - np.cumsum(T[..., :2], 1)) ** 2).sum(-1))[:, 19].mean()
    a, b, c2 = REF[nm]
    print(f"{nm:12s} {a:8d} {b:8d} {c2:8d} {e:8.0f}")